# 🏌️ DOH Vision — 백엔드 서버 (Colab GPU)

이 노트북은 **백엔드(FastAPI)** 를 콜랩 GPU에서 켜는 용도예요. 화면(프론트)은 **analyzer2** 가 담당.

**하는 법:** `런타임 ▸ 런타임 유형 변경 ▸ T4 GPU`, 그다음 `런타임 ▸ 모두 실행`.
맨 아래에 **공개 URL(https://…trycloudflare.com)** 이 뜨면, analyzer2의 `☁️ GPU 서버` 칸에 붙여넣으면 끝.

### 1칸 — 준비 (설치 + 모델·서버코드 받기)
`>>> 준비 완료` 뜨면 성공. 처음 한 번만 오래(모델 470MB).

In [ ]:
# 1칸 · 준비: FastAPI 설치 + 서버코드/엔진/모델 받기
!pip -q install fastapi "uvicorn[standard]" python-multipart 2>/dev/null
import os, urllib.request, torch, torchvision, torchvision.ops
print("torch", torch.__version__, "| GPU", torch.cuda.is_available())
BR  = "claude/session-context-recovery-qrdpov"
RAW = f"https://raw.githubusercontent.com/tinyalex3628-dotcom/doh-golf-survey/{BR}"
os.makedirs("pose3d_poc", exist_ok=True); os.makedirs("server", exist_ok=True)
for p in ["pose3d_poc/wham_golf_rotation.py", "pose3d_poc/wham_golf_metrics.py",
          "server/app.py", "server/index.html"]:
    urllib.request.urlretrieve(f"{RAW}/{p}", p)
M = "/content/nlf_l_multi_0.3.2.torchscript"   # 미리 받아 서버가 재다운로드 안 하게
if (not os.path.exists(M)) or os.path.getsize(M) < 10_000_000:
    print("NLF 모델 다운로드(470MB)…")
    urllib.request.urlretrieve("https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript", M)
os.environ["NLF_MODEL"] = M
print(">>> 준비 완료")

### 2칸 — 서버 실행 (공개 URL 생성)
실행하면 맨 아래에 **https://…trycloudflare.com** 주소가 떠요. analyzer2 백엔드 칸에 붙여넣기.
**이 셀은 계속 실행 상태로 두세요** (끄면 서버가 꺼짐).

In [ ]:
# 2칸 · FastAPI 백엔드(server/app.py) + Cloudflare 터널 → 공개 URL
import threading, subprocess, time, re, os, urllib.request, uvicorn
# (1) uvicorn 백그라운드 — 계약: POST /v1/analyze/video, GET /v1/jobs/{id}
def _run(): uvicorn.run("server.app:app", host="0.0.0.0", port=8000, log_level="warning")
threading.Thread(target=_run, daemon=True).start(); time.sleep(4)
# (2) cloudflared quick tunnel (계정 불필요)
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)
proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m: url = m.group(0); break
print("\n" + "="*66)
print("✅ 백엔드 공개 URL:", url)
print("→ 이 주소를 analyzer2 '☁️ GPU 서버' 칸에 붙여넣고 [서버로 분석].")
print("   이 셀은 계속 켜두세요(끄면 서버 꺼짐). 사지방서 trycloudflare 막히면 알려주세요.")
print("="*66)
for _ in proc.stdout: pass   # 서버 유지(셀 계속 실행)